In [1]:
# Parameters
TIME_TAG = 20251108071543


## 1. Preliminary

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os

USE_CUDF = False
try:
    # zero/low-code GPU acceleration for DataFrame ops
    os.environ["CUDF_PANDAS_BACKEND"] = "cudf"
    import pandas as pd
    import numpy as np

    USE_CUDF = True
    print("using cuda_backend pandas for faster parallel data processing")
except Exception:
    print("cuda df not used")
    import pandas as pd
    import numpy as np


from pathlib import Path
from sklearn.model_selection import GroupKFold
import warnings

warnings.filterwarnings("ignore")

using cuda_backend pandas for faster parallel data processing


In [4]:
from src.config import Config
from src.utils import (
    set_seed,
    load_input_output,
    write_meta,
    write_cv_log,
    load_saved_ensemble_stt,
)
from src.preprocess import prepare_sequences_with_advanced_features
from src.model import train_all_folds_stt
from src.predict import predict_sst

In [5]:
try:
    Config.TIME_TAG = str(TIME_TAG)[:8] + "_" + str(TIME_TAG)[8:]
    Config.SAVE_DIR = Path(f"./output/{Config.TIME_TAG}")
except NameError:
    Config.TIME_TAG = "default"

In [6]:
print(f"Current timetag: {Config.TIME_TAG}")

Current timetag: 20251108_071543


## 2. Train

In [8]:
def train():
   # 0) prepare output directory
   Config.SAVE_DIR.mkdir(parents=True, exist_ok=True)
   print(f"\n[0/4] Output directory: {Config.SAVE_DIR}")

   # 1) load training data
   print("\n[1/4] Loading training data")
   train_input, train_output = load_input_output()

   # 2) features + sequences
   print("\n[2/4] Feature Engineering")
   feature_groups = Config.FEATURE_GROUPS
   seqs, tdx, tdy, tfids, seq_meta, feat_cols = (
       prepare_sequences_with_advanced_features(
           train_input,
           output_df=train_output,
           feature_groups=feature_groups,
       )
   )
   input_dim = seqs[0].shape[1]

   sequences = list(seqs)
   targets_dx = list(tdx)
   targets_dy = list(tdy)

   # write meta
   write_meta(feat_cols, base_dir=Config.SAVE_DIR)

   # 3) multi-seed × KFold, save per-fold artifacts
   print("\n[3/4] Training model...")
   groups = np.array([f"{d['game_id']}_{d['play_id']}" for d in seq_meta])
   unique_grps, groups = np.unique(groups, return_inverse=True)
   print(f"Created {len(unique_grps)} unique groups (game_play_id) for GroupKFold.")

   seeds = Config.SEEDS
   all_rmse = []
   cv_log = []

   for seed in seeds:
       print(f"\n{'='*70}\n   Seed {seed}\n{'='*70}")
       set_seed(seed)
       gkf = GroupKFold(n_splits=Config.N_FOLDS)

       _all_rmse, _cv_log = train_all_folds_stt(
           gkf, sequences, groups, targets_dx, targets_dy, seed, input_dim
       )

       all_rmse.extend(_all_rmse)
       cv_log.extend(_cv_log)

   print()
   print(f"[CV SUMMARY] all folds RMSEs: {[f'{r:.4f}' for r in all_rmse]}")
   print(f"[CV SUMMARY] overall mean RMSE = {float(np.mean(all_rmse)):.4f} yards")

   write_cv_log(cv_log, all_rmse)
   return


if __name__ == "__main__":
   if Config.TRAIN:
       train()


[0/4] Output directory: output/20251108_071543

[1/4] Loading training data
Filtered input rows:  338
Filtered output rows:  752

[2/4] Feature Engineering

PREPARING SEQUENCES WITH ADVANCED FEATURES (UNIFIED FRAME)
Window size: 10
[21:48:06] [+] Added 'target_alignment' (4 cols) in 0.34s
    Columns: ['velocity_alignment', 'velocity_perpendicular', 'accel_alignment', 'accel_perpendicular']
[21:48:17] [+] Added 'lag' (24 cols) in 10.96s
    Columns: ['velocity_x_lag1', 'velocity_x_diff_lag1', 'velocity_y_lag1', 'velocity_y_diff_lag1', 's_lag1', 's_diff_lag1', 'velocity_x_lag2', 'velocity_x_diff_lag2', 'velocity_y_lag2', 'velocity_y_diff_lag2', 's_lag2', 's_diff_lag2', 'velocity_x_lag3', 'velocity_x_diff_lag3', 'velocity_y_lag3', 'velocity_y_diff_lag3', 's_lag3', 's_diff_lag3', 'velocity_x_lag5', 'velocity_y_lag5', 's_lag5', 'velocity_x_lag10', 'velocity_y_lag10', 's_lag10']
[21:48:20] [+] Added 'motion_change' (6 cols) in 2.66s
    Columns: ['velocity_x_change', 'velocity_y_change', '

  0%|          | 0/173137 [00:00<?, ?it/s]

[21:49:13] [+] Added 'curvature' (4 cols) in 23.57s
    Columns: ['bearing_to_land_signed', 'land_lateral_offset', 'curvature_signed', 'curvature_abs']
[21:49:28] [+] Added 'route' (8 cols) in 14.74s
    Columns: ['speed_mean', 'speed_change', 'traj_straightness', 'traj_depth', 'traj_width', 'traj_direction_angle', 'traj_max_turn', 'traj_mean_turn']
[21:49:37] [+] Added 'receiver' (4 cols) in 9.11s
    Columns: ['receiver_distance', 'v_to_receiver_alignment', 'v_to_receiver_perp', 'bearing_to_receiver']

Total features created: 127


Creating sequences (groups):   0%|          | 0/46037 [00:00<?, ?it/s]

Created 46037 sequences with 127 features each
Time to build sequences: 214.70 seconds
[META] wrote meta.json to output/20251108_071543

[3/4] Training model...
Created 14107 unique groups (game_play_id) for GroupKFold.

   Seed 42

------------------------------------------------------------
Fold 1/5 (seed 42)
------------------------------------------------------------
  Epoch  10: train=0.1160, val=0.0879, Time_elapsed= 0min 27s
  Epoch  20: train=0.0953, val=0.0817, Time_elapsed= 0min 53s
  Epoch  30: train=0.0910, val=0.0798, Time_elapsed= 1min 18s
  Epoch  40: train=0.0824, val=0.0768, Time_elapsed= 1min 44s
  Epoch  50: train=0.0775, val=0.0750, Time_elapsed= 2min 10s
  Epoch  60: train=0.0748, val=0.0741, Time_elapsed= 2min 36s
  Epoch  70: train=0.0736, val=0.0734, Time_elapsed= 3min  3s
  Epoch  80: train=0.0723, val=0.0734, Time_elapsed= 3min 31s
  Epoch  90: train=0.0720, val=0.0731, Time_elapsed= 3min 58s
  Epoch 100: train=0.0724, val=0.0732, Time_elapsed= 4min 24s
  Earl

## 3. Predict

In [ ]:
# =============================================================================
# Evaluation API Server Setup
# =============================================================================
# New imports for evaluation API
import polars as pl
from src.utils import load_saved_ensemble_stt, invert_to_original_direction
from src.preprocess import prepare_sequences_with_advanced_features
from src.model import STTransformer
from src.predict import predict_sst

# Global variables to store models (loaded once on first predict call)
_models_loaded = False
_models = None
_scalers = None
_meta = None
_feature_cols = None


def load_models_once():
    """Load models on first predict call (no 5-minute time limit)"""
    global _models_loaded, _models, _scalers, _meta, _feature_cols

    if _models_loaded:
        return

    print("[SERVER] Loading models for first time...")
    cfg = Config()
    cfg.MODELS_DIR = Path(
        f"/kaggle/input/nfl2026/{TIMETAG}"
    )  # pyright: ignore[reportUndefinedVariable]

    _models, _scalers, _meta = load_saved_ensemble_stt(cfg.MODELS_DIR, STTransformer)
    _feature_cols = _meta["feature_cols"]

    _models_loaded = True
    print(f"[SERVER] Loaded {len(_models)} models successfully")


def predict(
    test: pl.DataFrame, test_input: pl.DataFrame
) -> pl.DataFrame | pd.DataFrame:
    """
    Inference function: process each batch of data

    Args:
        test: Frames to predict (contains game_id, play_id, nfl_id, frame_id, etc.)
        test_input: Available input data (historical frames)

    Returns:
        DataFrame with x, y coordinates
    """
    global _models, _scalers, _meta, _feature_cols

    # First call: load models (no time limit)
    if not _models_loaded:
        load_models_once()

    # Convert to pandas (our code is pandas-based)
    test_pd = test.to_pandas()
    test_input_pd = test_input.to_pandas()

    cfg = Config()
    saved_groups = _meta.get("feature_groups", cfg.FEATURE_GROUPS)

    # Build sequences
    test_seqs, test_meta, feat_cols_t = prepare_sequences_with_advanced_features(
        test_input_pd,
        test_pd,
        feature_groups=saved_groups,
    )

    idx_x = feat_cols_t.index("x")
    idx_y = feat_cols_t.index("y")

    X_test_raw = list(test_seqs)
    x_last_uni = np.array([s[-1, idx_x] for s in X_test_raw], dtype=np.float32)
    y_last_uni = np.array([s[-1, idx_y] for s in X_test_raw], dtype=np.float32)

    all_preds_dx, all_preds_dy = [], []
    for m, sc in zip(_models, _scalers):
        dx_tta, dy_tta = predict_sst(
            m,
            sc,
            X_test_raw,
            cfg.DEVICE,
        )
        all_preds_dx.append(dx_tta)
        all_preds_dy.append(dy_tta)

    ens_dx = np.mean(all_preds_dx, axis=0)
    ens_dy = np.mean(all_preds_dy, axis=0)

    H = ens_dx.shape[1]

    # Build predictions
    rows = []
    tt_idx = test_pd.set_index(["game_id", "play_id", "nfl_id"]).sort_index()

    for i, meta_row in enumerate(test_meta):
        gid = meta_row["game_id"]
        pid = meta_row["play_id"]
        nid = meta_row["nfl_id"]
        play_dir = meta_row["play_direction"]

        try:
            fids = tt_idx.loc[(gid, pid, nid), "frame_id"]
            if isinstance(fids, pd.Series):
                fids = fids.sort_values().tolist()
            else:
                fids = [int(fids)]
        except KeyError:
            continue

        for t, fid in enumerate(fids):
            tt = min(t, H - 1)
            x_uni = np.clip(x_last_uni[i] + ens_dx[i, tt], 0, Config.FIELD_X_MAX)
            y_uni = np.clip(y_last_uni[i] + ens_dy[i, tt], 0, Config.FIELD_Y_MAX)
            x_uni, y_uni = invert_to_original_direction(
                x_uni, y_uni, play_dir == "right"
            )
            rows.append({"x": x_uni, "y": y_uni})

    predictions = pl.DataFrame(rows)

    assert len(predictions) == len(test)
    return predictions


if Config.SUBMIT:
    import kaggle_evaluation.nfl_inference_server  # type: ignore

    # Initialize inference server
    inference_server = kaggle_evaluation.nfl_inference_server.NFLInferenceServer(
        predict
    )

    # Start server in competition environment
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        print("[SERVER] Starting inference server...")
        inference_server.serve()
    else:
        print("[SERVER] Running local gateway for testing...")
        inference_server.run_local_gateway(
            ("/kaggle/input/nfl-big-data-bowl-2026-prediction/",)
        )